# HybridTokenizer Training Notebook

Trains and saves your custom `HybridTokenizer` on real text corpora.

**What this notebook does:**
1. Feeds text from WikiText-2 + TinyStories into the tokenizer's frequency database  
2. Calls `freeze_vocab()` to finalize base tokens + PMI-scored merge rules  
3. Tests round-trip encode/decode correctness  
4. Saves the tokenizer to `tokenizer.pkl.gz`  

**After running:** copy `tokenizer.pkl.gz` to your Drive or download it — you need it for `slm.ipynb`.

In [ ]:
%pip install -q "git+https://github.com/sh20022002/small-Language-Model.git@main"
%pip install -q datasets

In [ ]:
import os, time
from datasets import load_dataset
from my_slm.hybrid_tokeniztion import HybridTokenizer

SAVE_PATH = '/content/tokenizer.pkl.gz'   # change if you want Drive storage

## Step 1 — Build the word frequency database

The tokenizer learns which words appear most in your training corpus.  
More diverse text → better vocabulary coverage.  
We use two small datasets that together take ~30 seconds to scan.

In [ ]:
print('Loading datasets...')
wiki  = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
wiki  = wiki.filter(lambda x: len(x['text'].strip()) > 20)

# TinyStories: short children stories — great for common English vocabulary
stories = load_dataset('roneneldan/TinyStories', split='train').select(range(50_000))

print(f'WikiText-2 train samples : {len(wiki):,}')
print(f'TinyStories samples      : {len(stories):,}')

In [ ]:
tok = HybridTokenizer(lowercase=True, byte_limit=4)

t0 = time.time()

print('Scanning WikiText-2...')
for ex in wiki:
    tok.add_text(ex['text'])

print('Scanning TinyStories...')
for ex in stories:
    tok.add_text(ex['text'])

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s  |  unique word types: {len(tok.word_db):,}')

In [ ]:
# Check coverage before freezing — if base_ratio < 90% consider raising byte_limit
tok.db_status(preview=15, k_bases=2000)

## Step 2 — Freeze the vocabulary

Parameters:
- `k_bases`: how many single words to keep as base tokens (top-k by frequency)
- `max_merges`: how many word-pair merges to learn (scored by PMI × log-freq)

Final vocab size ≈ 9 special + 256 byte fallbacks + k_bases + merges (≈ 12 000–13 000)

In [ ]:
t0 = time.time()
tok.freeze_vocab(k_bases=2000, max_merges=8000, min_freq=3)
print(f'Frozen in {time.time()-t0:.1f}s')
print(f'Vocab size: {tok.vocab_size:,}')
print(f'  special tokens : {len(tok.special_tokens)}')
print(f'  byte fallbacks : 256')
print(f'  other tokens   : {tok.vocab_size - len(tok.special_tokens) - 256}')

## Step 3 — Verify correctness

In [ ]:
# Round-trip test: encode then decode must reproduce original text
tok.self_test()

# Extra round-trip checks
test_sentences = [
    'The quick brown fox jumps over the lazy dog.',
    'Neural networks learn representations from data.',
    'Hello, world! 🌍  \nNew line here.',
    '1 + 1 = 2  (arithmetic)',
]
for s in test_sentences:
    ids   = tok.encode(s, mode='flat')
    back  = tok.decode(ids)
    match = '✓' if back == s else '✗'
    print(f'{match}  [{len(ids):3d} tokens]  {s[:60]}')
    if back != s:
        print(f'   Expected : {repr(s)}')
        print(f'   Got      : {repr(back)}')

In [ ]:
# Inspect how the tokenizer breaks up words
examples = [
    'The capital of France is Paris.',
    'Transformers use self-attention mechanisms.',
    'running runners ran',
]
for text in examples:
    print(f'\nInput: {text}')
    segments = tok.segment(text)
    for token_str, kind in segments:
        print(f'  {token_str:20}  [{kind}]')

In [ ]:
# Show top learned merges (PMI × frequency)
print('Top 30 learned merges:')
tok.top_merges(30)

In [ ]:
# Compression ratio: how many tokens per character on average?
sample_text = ' '.join(ex['text'] for ex in wiki.select(range(100)))
ids = tok.encode(sample_text, mode='flat')
chars = len(sample_text)
tokens = len(ids)
print(f'Sample text length : {chars:,} chars')
print(f'Encoded length     : {tokens:,} tokens')
print(f'Compression ratio  : {chars/tokens:.2f} chars/token')
print(f'(BERT averages ~4-5 chars/token; GPT-2 averages ~3-4)')

## Step 4 — Save the tokenizer

In [ ]:
tok.save(SAVE_PATH)
size_mb = os.path.getsize(SAVE_PATH) / 1e6
print(f'Saved to {SAVE_PATH}  ({size_mb:.1f} MB)')

# Verify: reload and check vocab_size matches
tok2 = HybridTokenizer.load(SAVE_PATH)
assert tok2.vocab_size == tok.vocab_size, 'Vocab size mismatch after reload!'
assert tok2.frozen, 'Tokenizer should be frozen after reload'
print(f'Reload OK  |  vocab_size={tok2.vocab_size:,}')

In [ ]:
# Optional: copy to Google Drive so it persists across sessions
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(SAVE_PATH, '/content/drive/MyDrive/slm/tokenizer.pkl.gz')
# print('Copied to Drive')

## How to use the saved tokenizer in `slm.ipynb`

```python
from my_slm.hybrid_tokeniztion import HybridTokenizer

tok = HybridTokenizer.load('/content/tokenizer.pkl.gz')
pad_id    = tok.token2id['<PAD>']
eos_id    = tok.token2id['<EOS>']
vocab_size = tok.vocab_size

# Encode text
ids = tok.encode('Hello world', mode='flat')   # list[int]

# Decode
text = tok.decode(ids)                          # str
```